# Linux Kernel Vulnerability Insights: Data Exploration

This notebook provides an interactive walkthrough of the kernel bug-fix pairs dataset.

**What you'll learn:**
- How to load and explore the dataset
- Key patterns in vulnerability lifetimes
- Who catches bugs fastest (and why)
- Temporal and subsystem patterns

**Prerequisites:**
```bash
pip install -r requirements.txt
python scripts/download_data.py
```

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

# Project utilities
import sys
sys.path.insert(0, '../scripts')
from utils import (
    load_dataset, compute_lifetime_stats, identify_super_reviewers,
    compute_subsystem_stats, compute_temporal_trends, compute_weekday_effect,
    COLOR_PALETTE
)

## 1. Loading the Dataset

The dataset contains 125,000+ bug-fix pairs extracted from the Linux kernel git history using `Fixes:` tags.

In [ ]:
# Load the dataset
df = load_dataset()
print(f"Loaded {len(df):,} bug-fix pairs")
print(f"\nDate range: {df['buggy_date'].min().date()} to {df['fix_date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Preview the data
df.head()

In [ ]:
# Basic statistics
df.describe()

## 2. Bug Lifetime Analysis

The central question: **How long do security bugs survive in the kernel before being fixed?**

In [ ]:
# Compute lifetime statistics
stats = compute_lifetime_stats(df)

print("Bug Lifetime Statistics")
print("=" * 40)
print(f"Total bugs analyzed: {stats['count']:,}")
print(f"\nCentral tendency:")
print(f"  Median: {stats['median_years']:.1f} years ({stats['median_days']:.0f} days)")
print(f"  Mean:   {stats['mean_years']:.1f} years ({stats['mean_days']:.0f} days)")
print(f"\nSpread:")
print(f"  Std dev: {stats['std_days']:.0f} days")
print(f"  Min:     {stats['min_days']:.0f} days")
print(f"  Max:     {stats['max_years']:.1f} years ({stats['max_days']:.0f} days)")
print(f"\nPercentiles:")
print(f"  25th: {stats['p25_days']/365:.1f} years")
print(f"  75th: {stats['p75_days']/365:.1f} years")
print(f"  90th: {stats['p90_days']/365:.1f} years")
print(f"  99th: {stats['p99_days']/365:.1f} years")

In [ ]:
# Visualize the distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lifetime_years = df['lifetime_days'] / 365

# Histogram
ax1 = axes[0]
ax1.hist(lifetime_years, bins=100, color=COLOR_PALETTE['primary'], 
         alpha=0.7, edgecolor='white', linewidth=0.5)
ax1.axvline(lifetime_years.median(), color=COLOR_PALETTE['secondary'], 
            linestyle='--', linewidth=2, label=f'Median: {lifetime_years.median():.1f} years')
ax1.set_xlabel('Bug Lifetime (years)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Bug Lifetimes')
ax1.legend()
ax1.set_xlim(0, 15)

# CDF
ax2 = axes[1]
sorted_lifetime = np.sort(lifetime_years)
cdf = np.arange(1, len(sorted_lifetime) + 1) / len(sorted_lifetime)
ax2.plot(sorted_lifetime, cdf, color=COLOR_PALETTE['primary'], linewidth=2)
ax2.axhline(0.5, color=COLOR_PALETTE['neutral'], linestyle=':', alpha=0.5)
ax2.axvline(lifetime_years.median(), color=COLOR_PALETTE['secondary'], linestyle='--', linewidth=2)
ax2.set_xlabel('Bug Lifetime (years)')
ax2.set_ylabel('Cumulative Proportion')
ax2.set_title('Cumulative Distribution')
ax2.set_xlim(0, 20)

plt.tight_layout()
plt.show()

print(f"\n Key insight: The median bug survives {lifetime_years.median():.1f} years,")
print(f"   but the long tail extends to {lifetime_years.max():.1f} years!")

## 3. Super-Reviewer Analysis

Some reviewers consistently catch bugs faster than others. We call them "super-reviewers."

**Definition**: A developer who has reviewed 50+ bug fixes and whose median bug lifetime is below the overall median.

In [ ]:
# Identify super-reviewers
super_reviewers, speedup = identify_super_reviewers(df, min_fixes=50)

print(f"Found {len(super_reviewers)} super-reviewers")
print(f"They catch bugs {speedup:.0%} faster than average!")
print(f"\nTop 10 super-reviewers:")
super_reviewers.head(10)[['reviewer', 'fix_count', 'median_lifetime', 'speedup']]

In [ ]:
# Why 50+ fixes as the threshold?
# Let's see how the effect changes with different thresholds

thresholds = [10, 20, 30, 50, 75, 100]
results = []

for thresh in thresholds:
    sr, sp = identify_super_reviewers(df, min_fixes=thresh)
    results.append({
        'threshold': thresh,
        'num_super_reviewers': len(sr),
        'speedup': sp
    })

threshold_df = pd.DataFrame(results)
print("Sensitivity to threshold choice:")
print(threshold_df.to_string(index=False))
print("\n The speedup effect is robust across thresholds (42-51%)")

## 4. Subsystem Analysis

Do some parts of the kernel have longer-lived bugs than others?

In [ ]:
# Compute subsystem statistics
subsystem_stats = compute_subsystem_stats(df)
print("Top 15 subsystems by bug count:")
subsystem_stats.head(15)[['subsystem', 'bug_count', 'median_lifetime', 'lifetime_years']]

In [ ]:
# Visualize subsystem differences
fig, ax = plt.subplots(figsize=(12, 8))

top_subsystems = subsystem_stats.head(15)
y_pos = np.arange(len(top_subsystems))

bars = ax.barh(y_pos, top_subsystems['lifetime_years'], 
               color=COLOR_PALETTE['primary'], alpha=0.8)

# Add overall median
overall_median = df['lifetime_days'].median() / 365
ax.axvline(overall_median, color=COLOR_PALETTE['secondary'], 
           linestyle='--', linewidth=2, label=f'Overall median: {overall_median:.1f}y')

ax.set_yticks(y_pos)
ax.set_yticklabels(top_subsystems['subsystem'])
ax.set_xlabel('Median Bug Lifetime (years)')
ax.set_title('Bug Lifetime by Kernel Subsystem')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Temporal Patterns

How have bug lifetimes changed over 20 years?

In [ ]:
# Compute trends
trends = compute_temporal_trends(df, freq='Y')

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Bug introduction rate
ax1 = axes[0]
ax1.bar(trends['period_start'], trends['bug_count'], 
        width=300, color=COLOR_PALETTE['primary'], alpha=0.7)
ax1.set_ylabel('Bugs Introduced')
ax1.set_title('Kernel Bug Introduction and Lifetime Trends')

# Median lifetime
ax2 = axes[1]
ax2.plot(trends['period_start'], trends['median_lifetime'] / 365, 
         marker='o', linewidth=2, color=COLOR_PALETTE['secondary'])
ax2.fill_between(trends['period_start'], 0, trends['median_lifetime'] / 365,
                 alpha=0.2, color=COLOR_PALETTE['secondary'])
ax2.set_xlabel('Year')
ax2.set_ylabel('Median Bug Lifetime (years)')

plt.tight_layout()
plt.show()

print(" Bug introduction rate is increasing, but median lifetimes remain stable.")
print("   This suggests detection methods are keeping pace with code growth.")

## 6. The Weekend Effect

A surprising finding: bugs introduced on weekends survive longer!

In [ ]:
# Analyze day-of-week patterns
weekday_stats = compute_weekday_effect(df)

fig, ax = plt.subplots(figsize=(10, 6))

colors = [COLOR_PALETTE['secondary'] if day in ['Saturday', 'Sunday'] 
          else COLOR_PALETTE['primary'] 
          for day in weekday_stats['weekday']]

bars = ax.bar(weekday_stats['weekday'], 
              weekday_stats['median_lifetime'] / 365,
              color=colors, alpha=0.8, edgecolor='white', linewidth=1)

# Add value labels
for bar, val in zip(bars, weekday_stats['median_lifetime'] / 365):
    ax.annotate(f'{val:.2f}y',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=10)

overall_median = df['lifetime_days'].median() / 365
ax.axhline(overall_median, color=COLOR_PALETTE['neutral'], 
           linestyle='--', linewidth=2, alpha=0.7,
           label=f'Overall median: {overall_median:.2f}y')

ax.set_xlabel('Day Bug Was Introduced')
ax.set_ylabel('Median Bug Lifetime (years)')
ax.set_title('The Weekend Effect: Bugs Introduced on Weekends Survive Longer')
ax.legend()

plt.tight_layout()
plt.show()

# Calculate effect size
weekday_median = weekday_stats[~weekday_stats['weekday'].isin(['Saturday', 'Sunday'])]['median_lifetime'].mean()
weekend_median = weekday_stats[weekday_stats['weekday'].isin(['Saturday', 'Sunday'])]['median_lifetime'].mean()
effect = (weekend_median - weekday_median) / weekday_median * 100

print(f"\n Weekend commits have {effect:.0f}% longer bug lifetime!")
print("   Possible explanations:")
print("   - Less review attention on weekend patches")
print("   - Different developer population (hobbyists vs. professionals)")
print("   - Rushed commits before Monday deadlines")

## 7. Security Bug Analysis

Security-relevant bugs are particularly important. How do they compare?

In [ ]:
# Security bug breakdown
if 'is_security' in df.columns:
    security_bugs = df[df['is_security'] == True]
    other_bugs = df[df['is_security'] == False]
    
    print(f"Security bugs: {len(security_bugs):,} ({100*len(security_bugs)/len(df):.1f}%)")
    print(f"Other bugs: {len(other_bugs):,} ({100*len(other_bugs)/len(df):.1f}%)")
    
    print(f"\nMedian lifetime:")
    print(f"  Security bugs: {security_bugs['lifetime_days'].median()/365:.2f} years")
    print(f"  Other bugs:    {other_bugs['lifetime_days'].median()/365:.2f} years")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    
    data = [
        security_bugs['lifetime_days'] / 365,
        other_bugs['lifetime_days'] / 365
    ]
    
    bp = ax.boxplot(data, labels=['Security Bugs', 'Other Bugs'], patch_artist=True)
    bp['boxes'][0].set_facecolor(COLOR_PALETTE['security'])
    bp['boxes'][1].set_facecolor(COLOR_PALETTE['primary'])
    
    ax.set_ylabel('Bug Lifetime (years)')
    ax.set_title('Security vs. Non-Security Bug Lifetimes')
    ax.set_ylim(0, 10)
    
    plt.tight_layout()
    plt.show()
else:
    print("'is_security' column not found in dataset")

## 8. Key Takeaways

Let's summarize what we've learned:

In [ ]:
# Summary statistics
print("="*60)
print("LINUX KERNEL VULNERABILITY INSIGHTS - KEY FINDINGS")
print("="*60)
print(f"\n Dataset: {len(df):,} bug-fix pairs over 20 years")
print(f"\n Bug Lifetimes:")
print(f"    Median: {df['lifetime_days'].median()/365:.1f} years")
print(f"    Mean: {df['lifetime_days'].mean()/365:.1f} years")
print(f"    Max: {df['lifetime_days'].max()/365:.1f} years")
print(f"\n Super-Reviewers:")
print(f"    Found {len(super_reviewers)} developers")
print(f"    Catch bugs {speedup:.0%} faster than average")
print(f"\n Weekend Effect:")
print(f"    Weekend commits have {effect:.0f}% longer bug lifetime")
print(f"\n Top Subsystems by Bug Count:")
for _, row in subsystem_stats.head(5).iterrows():
    print(f"    {row['subsystem']}: {row['bug_count']:,} bugs, {row['lifetime_years']:.1f}y median")
print("\n" + "="*60)

## Next Steps

Now that you've explored the data, you can:

1. **Generate publication-quality figures**: `python scripts/generate_all_plots.py`
2. **Explore temporal patterns**: See `notebooks/02_temporal_analysis.ipynb`
3. **Deep-dive on reviewers**: See `notebooks/03_reviewer_analysis.ipynb`
4. **Subsystem-specific analysis**: See `notebooks/04_subsystem_patterns.ipynb`

For methodology details, see:
- `methodology/DATA_CONSTRUCTION.md` - How the dataset was built
- `methodology/ANALYSIS_CHOICES.md` - Why we made specific decisions
- `methodology/LIMITATIONS.md` - What this analysis doesn't capture